In [1]:
import os
import pandas as pd
# use polars when using very large dataframes as polars is faster than pandas
import polars as pl

import seaborn as sns

# use pickle to save and load data objects (when saving a df using pickle, you save more information)
import pickle
import matplotlib.pyplot as plt
from datetime import datetime

import matplotlib.patches as mpatches

from itertools import product


# With the ic() method you can print something that is between the brackets and automatically also print its name
from icecream import ic

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


In [2]:
# Read the Parquet file
df_scoped_pl = pl.read_parquet("C:/Users/jasmi/Documents/2025_Data_analyse/2025_03_Group_project_Supermarket/Data/data/df_scope.parquet")
df_scoped = df_scoped_pl.to_pandas()
print(f'the number of columns is {len(df_scoped.columns)}')

# Display column names, their data types, and the number of unique values:
print("Column Names and Data Types are:")
for column in df_scoped.columns:
 print(f"{column} datatype is: {df_scoped[column].dtype}")
 print(f'The number of unique values of {column} is: {len(df_scoped[column].unique())}')



the number of columns is 24
Column Names and Data Types are:
id datatype is: int32
The number of unique values of id is: 1993256
store_nbr datatype is: int8
The number of unique values of store_nbr is: 8
item_nbr datatype is: float32
The number of unique values of item_nbr is: 231
unit_sales datatype is: float32
The number of unique values of unit_sales is: 328
onpromotion datatype is: object
The number of unique values of onpromotion is: 3
day datatype is: int8
The number of unique values of day is: 31
month datatype is: category
The number of unique values of month is: 56
family datatype is: category
The number of unique values of family is: 1
class datatype is: int16
The number of unique values of class is: 22
perishable datatype is: int8
The number of unique values of perishable is: 1
city datatype is: category
The number of unique values of city is: 3
state datatype is: category
The number of unique values of state is: 3
type_x datatype is: category
The number of unique values of 

## Remove the oldest data from the most sold item and the 2013 data from all items

In [3]:
# Remove most sold item
top_item = df_scoped.groupby('item_nbr')['unit_sales'].sum().idxmax()
mask = (df_scoped['item_nbr'] == top_item) & (df_scoped['date'] < '2015-06-01')
df_filtered = df_scoped[~mask]

# Remove all 2013 data
df_scoped_3 = df_filtered[df_filtered['date'] > datetime(2013, 12, 31)].copy()
print(f"the number of items in the originala scoped df is {df_scoped['item_nbr'].nunique()}")
print(f"the number of items in df_scoped_3 (=df with adapted top_item and without 2013 data) is {df_scoped_3['item_nbr'].nunique()}")

print(f"the number of unique store-item combinations in the scoped df is: {df_scoped[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]}")
print(f"the number of unique store-item combinations in df_scoped_3 (=df with adapted top_item and without 2013 data) df is: {df_scoped_3[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]}")

the number of items in the originala scoped df is 231
the number of items in df_scoped_3 (=df with adapted top_item and without 2013 data) is 231
the number of unique store-item combinations in the scoped df is: 1833
the number of unique store-item combinations in df_scoped_3 (=df with adapted top_item and without 2013 data) df is: 1832


## Remove store-item combinations that have no recent sales AND have been historically sparse

In [4]:
# Find out the most recent date in the df
latest_date = pd.to_datetime(df_scoped_3['date']).max()

#  Compute sales in the last 30 days
last_30_mask = df_scoped_3['date'] >= (latest_date - pd.Timedelta(days=30))
sales_last_30 = (
    df_scoped_3[last_30_mask]
    .groupby(['store_nbr', 'item_nbr'])['unit_sales']
    .sum()
    .reset_index(name='sales_last_30_days')
)

# Compute number of active days in last 365 days (sales > 0)
last_365_mask = df_scoped_3['date'] >= (latest_date - pd.Timedelta(days=365))
active_days_last_365 = (
    df_scoped_3[last_365_mask & (df_scoped_3['unit_sales'] > 0)]
    .groupby(['store_nbr', 'item_nbr'])['date']
    .nunique()
    .reset_index(name='nbr_active_days_last_365')
)

# Merge these metrics back into your DataFrame
df_metrics = sales_last_30.merge(active_days_last_365, on=['store_nbr', 'item_nbr'], how='outer').fillna(0)

# Remove store–item combinations meeting your condition
bad_combos = df_metrics[
    (df_metrics['sales_last_30_days'] == 0) &
    (df_metrics['nbr_active_days_last_365'] < 60)
][['store_nbr', 'item_nbr']]

df_scoped_4 = df_scoped_3.merge(bad_combos, on=['store_nbr', 'item_nbr'], how='left', indicator=True)
df_scoped_4 = df_scoped_4[df_scoped_4['_merge'] == 'left_only'].drop(columns='_merge')

print(f"the number of items in df_scoped_4 is {df_scoped_4['item_nbr'].nunique()}")

print(f"the number of unique store-item combinations in df_scoped_4 is: {df_scoped_4[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]}")


the number of items in df_scoped_4 is 231
the number of unique store-item combinations in df_scoped_4 is: 1827


# Filter for at least 365 days span and remove combos that have not at least 12 consecutive weeks of sales data

In [5]:
# Get latest date for reference
latest_date = df_scoped_4['date'].max()

# Step 1: Calculate days span per store-item
sales_span = (df_scoped_4.groupby(['store_nbr', 'item_nbr'])['date']
               .agg(first_sale='min', last_sale='max')
               .reset_index())

sales_span['days_span'] = (sales_span['last_sale'] - sales_span['first_sale']).dt.days

# Filter for at least 365 days span
valid_days_span = sales_span[sales_span['days_span'] >= 365][['store_nbr', 'item_nbr']]

# Step 2: Filter to last 365 days & positive sales, add 'week' period
last_365_mask = df_scoped_4['date'] >= (latest_date - pd.Timedelta(days=365))
df_last_year = df_scoped_4[last_365_mask & (df_scoped_4['unit_sales'] > 0)].copy()
df_last_year['week'] = df_last_year['date'].dt.to_period('W')

# Step 3: Get sorted unique active weeks per store-item
active_weeks = (
    df_last_year.groupby(['store_nbr', 'item_nbr'])['week']
    .apply(lambda weeks: sorted(weeks.unique()))
    .reset_index(name='active_weeks_list')
)

# Helper function to check for >=12 consecutive weeks
def has_12_consecutive_weeks(weeks_list):
    if len(weeks_list) < 12:
        return False
    week_nums = [w.ordinal for w in weeks_list]
    count = 1
    for i in range(1, len(week_nums)):
        if week_nums[i] == week_nums[i-1] + 1:
            count += 1
            if count >= 12:
                return True
        else:
            count = 1
    return False

# Step 4: Filter for 12 consecutive active weeks
active_weeks['has_12_consecutive'] = active_weeks['active_weeks_list'].apply(has_12_consecutive_weeks)
valid_consecutive = active_weeks[active_weeks['has_12_consecutive']][['store_nbr', 'item_nbr']]

# Step 5: Merge both filters: days_span and consecutive weeks
valid_combos = pd.merge(valid_days_span, valid_consecutive, on=['store_nbr', 'item_nbr'], how='inner')

# Step 6: Filter original data to keep only valid store-item combinations
df_scoped_5 = df_scoped_4.merge(valid_combos, on=['store_nbr', 'item_nbr'], how='inner')

print(f"the number of items in df_scoped_5 is {df_scoped_5['item_nbr'].nunique()}")

print(f"the number of unique store-item combinations in df_scoped_5 is: {df_scoped_5[['store_nbr', 'item_nbr']].drop_duplicates().shape[0]}")



the number of items in df_scoped_5 is 208
the number of unique store-item combinations in df_scoped_5 is: 1631


## Opslaan als pickle file

In [6]:
# Create dictionary 'dc_scoped_df' with objects that will be used in the next parts.
dc_scoped_df = {
    'df_scoped_5':      df_scoped_5,
    }

# Save dc_scoped_df as 'dc-scoped-df.pkl'
with open('../Data/data/dc-scoped-df.pkl', 'wb') as pickle_file:
    pickle.dump(dc_scoped_df, pickle_file)